<a href="https://colab.research.google.com/github/abyanabdillah0-maker/Tugas-Github/blob/main/Regresi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis pengaruh belanja modal fiskal terhadap tingkat pengangguran di Kota Sibolga

Variabel yang Diukur:

*   Variabel Terikat (Y): (Tingkat Pengangguran)
*   Variabel Bebas (X1) : (Belanja Modal Fungsi Ekonomi)
*   Variabel Bebas (X2): (Belanja Modal Fungsi Perlindungan Sosial)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

# -----------------------------------------------------------------------------------------
# 1. MEMBACA DATA ASLI

url = 'https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv'
df_all = pd.read_csv(url)
df = df_all[df_all['Kota'] == 'Kota Sibolga'].copy()

print(f"Total data Sibolga yang dianalisis: {len(df)} baris.\n")

# -----------------------------------------------------------------------------------------
# 2. MODEL REGRESI & UJI ASUMSI KLASIK

X = df[['bm_ekonomi', 'bm_perlindungan_sosial']]
Y = df['PENGANGGURAN']
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI:")
print(f"Y = {model.params['const']:.4f} + {model.params['bm_ekonomi']:.4f}(BM_Ekonomi) - {abs(model.params['bm_perlindungan_sosial']):.4f}(BM_Sosial)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK")
# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)            : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# --------------------------------------------------------------------------------
# 3. VISUALISASI 3D INTERAKTIF

# Membuat grid bidang regresi berdasarkan rentang data asli
x_range = np.linspace(df['bm_ekonomi'].min(), df['bm_ekonomi'].max(), 20)
y_range = np.linspace(df['bm_perlindungan_sosial'].min(), df['bm_perlindungan_sosial'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang (DIPERBAIKI)
z_grid = model.params['const'] + (model.params['bm_ekonomi'] * x_grid) + (model.params['bm_perlindungan_sosial'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Asli
fig.add_trace(go.Scatter3d(
    x=df['bm_ekonomi'],
    y=df['bm_perlindungan_sosial'],
    z=df['PENGANGGURAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: Pengangguran terhadap BM Ekonomi & BM Sosial (Kota Sibolga) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Ekonomi',
        yaxis_title='BM Perlindungan Sosial',
        zaxis_title='Pengangguran'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()

Total data Sibolga yang dianalisis: 11 baris.

                            OLS Regression Results                            
Dep. Variable:           PENGANGGURAN   R-squared:                       0.704
Model:                            OLS   Adj. R-squared:                  0.629
Method:                 Least Squares   F-statistic:                     9.495
Date:                Fri, 16 Oct 2026   Prob (F-statistic):            0.00772
Time:                        21:01:15   Log-Likelihood:                -15.560
No. Observations:                  11   AIC:                             37.12
Df Residuals:                       8   BIC:                             38.31
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

# Menguji pengaruh satu variabel bebas secara spesifik terhadap tingkat pengangguran

Variabel yang Diukur:

*   Variabel Bebas / Independen (X): (Belanja Modal Fungsi Ekonomi)
*   Variabel Terikat / Dependen (Y): (Tingkat Pengangguran)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'colab'

# 0. Muat dataset utama dan bersihkan nama kolom
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.strip()

# FILTER: Hanya mengambil data khusus untuk Kota Sibolga
df_sibolga = df[df['Kota'] == 'Kota Sibolga'].copy()

# 1. Menentukan Variabel (1 Independen X, 1 Dependen Y) untuk Kota Sibolga
X_2v = df_sibolga[['bm_ekonomi']]
Y_2v = df_sibolga['PENGANGGURAN']

# Menambahkan konstanta dan melatih model
X_2v_sm = sm.add_constant(X_2v)
model_2v = sm.OLS(Y_2v, X_2v_sm).fit()

# 2. Output Ringkasan Model & Persamaan Regresi
print("="*50)
print("HASIL REGRESI 2 VARIABEL - KOTA SIBOLGA")
print("="*50)
print(model_2v.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI 2 VARIABEL (KOTA SIBOLGA):")
print(f"Y = {model_2v.params['const']:.4f} + {model_2v.params['bm_ekonomi']:.4f}(BM_Ekonomi)")
print("="*50)

# 3. Uji Asumsi Klasik
print("\nHASIL UJI ASUMSI KLASIK:")
try:
    shapiro_stat, shapiro_p = stats.shapiro(model_2v.resid)
    print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")
except Exception as e:
    print(f"- Normalitas (Shapiro-Wilk): {e}")

try:
    bp_test = sms.het_breuschpagan(model_2v.resid, model_2v.model.exog)
    print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")
except Exception as e:
    print(f"- Heteroskedastisitas (BP): {e}")

dw_value = sm.stats.stattools.durbin_watson(model_2v.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# 4. Visualisasi Grafik 2D Interaktif (Scatter Plot + Garis Regresi)
# PERBAIKAN: Menghapus tanda '-' di awal baris dan menambahkan operator '*' untuk perkalian
x_line = np.linspace(df_sibolga['bm_ekonomi'].min(), df_sibolga['bm_ekonomi'].max(), 100)
y_line = model_2v.params['const'] + (model_2v.params['bm_ekonomi'] * x_line)

fig_2d = go.Figure()

# Titik Data Aktual Kota Sibolga
fig_2d.add_trace(go.Scatter(
    x=df_sibolga['bm_ekonomi'],
    y=df_sibolga['PENGANGGURAN'],
    mode='markers+text',
    text=df_sibolga['Tahun'], # Menampilkan label tahun di setiap titik data
    textposition='top center',
    name='Data Aktual Sibolga',
    marker=dict(size=10, color='red', line=dict(color='white', width=1))
))

# Garis Regresi Linear
fig_2d.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    name='Garis Regresi',
    line=dict(color='#00CC96', width=3)
))

# Layout Tampilan 2D
fig_2d.update_layout(
    title=f"Plot Regresi 2D (Kota Sibolga): BM Ekonomi vs Pengangguran | R² = {model_2v.rsquared:.2f}",
    template='plotly_dark',
    xaxis_title='BM Ekonomi (X)',
    yaxis_title='Pengangguran (Y)',
    margin=dict(l=40, r=40, b=40, t=50)
)

fig_2d.show()

HASIL REGRESI 2 VARIABEL - KOTA SIBOLGA
                            OLS Regression Results                            
Dep. Variable:           PENGANGGURAN   R-squared:                       0.381
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     5.528
Date:                Fri, 16 Oct 2026   Prob (F-statistic):             0.0432
Time:                        21:04:06   Log-Likelihood:                -19.615
No. Observations:                  11   AIC:                             43.23
Df Residuals:                       9   BIC:                             44.03
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       